# Qwen3-4B-Thinking-2507 -- evaluate the ViNumQA SFT adapter (Modal)

Loads a LoRA adapter that was trained by `qwen3-4b-thinking-2507-stf-w-reasoning-trace-modal.ipynb`
and left on the Modal Volume (e.g. because the Modal session self-stopped after training but before
the eval loop finished) and scores Program Accuracy / Execution Accuracy on `test.json`, using the
same parser as every other notebook in this repo so the numbers stay comparable.

Cloned from `qwen3-4b-thinking-2507-eval-only.ipynb` (the Kaggle-only version, which pairs with the
separate Kaggle-only training notebook and expects a `/kaggle/input/...` adapter) -- this version
instead reads `ADAPTER_DIR`/`test.json`/checkpoints from the same Modal Volume (`/mnt/qwen3-4b-thinking-2507-sft`) the
training notebook used, since the adapter never left Modal.

`EVAL_TAG` below tags the eval checkpoint files (`eval_partial_k*_<tag>.csv`,
`retest_k5_results_<tag>.json`) with whichever adapter you're evaluating, same reasoning as
`DATASET_VARIANT` in the training notebook: if you later evaluate a different adapter on the same
Volume, the two runs' checkpoints won't collide or get resumed into each other.

### Installation

In [ ]:
%%capture
# IMPORTANT (real precedent on this repo's Modal setup, not theoretical):
# after this cell finishes, RESTART THE KERNEL (Kernel > Restart) before
# running anything below, then re-run from the top. Modal's Server base image
# ships older typing_extensions/pydantic versions that other packages import
# into the running process BEFORE this cell upgrades them on disk -- the
# upgrade doesn't unload what's already in memory, so `from unsloth import`
# below can fail (e.g. `ImportError: cannot import name 'Sentinel' from
# 'typing_extensions'`) unless the kernel restarts first. This is why `Run
# All` in one uninterrupted pass is not safe here -- run this cell, restart,
# THEN Run All from the top (the pip installs are idempotent, so re-running
# this cell after restart costs a little time, not correctness).
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups (incl. Kaggle)
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install -q tabulate sympy
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### HF cache (Modal Volume)

In [ ]:
import os
from pathlib import Path

# Point the HuggingFace cache at the same Modal Volume the training notebook used,
# so the base model (already downloaded once during training) is reused instead of
# re-downloaded. Must run before any `from unsloth import ...` / `from transformers
# import ...` below (env vars are read at import time).
HF_CACHE = Path("/mnt/qwen3-4b-thinking-2507-sft")
os.environ["HF_HUB_CACHE"] = str(HF_CACHE / "hf_cache")

if not HF_CACHE.exists():
    print(f"!! {HF_CACHE} does not exist -- no Volume attached at that path.")
    print("   Attach the same Volume the training notebook used via the sidebar before continuing,")
    print("   otherwise the base model re-downloads to ephemeral disk (works, just slower).")
else:
    print(f"Volume attached at {HF_CACHE} -- HF cache -> {HF_CACHE / 'hf_cache'}")

### Load the fine-tuned adapter

In [ ]:
from unsloth import FastLanguageModel
import torch
import glob, os

MAX_SEQ_LENGTH = 8192

# Adapter left on the Modal Volume by qwen3-4b-thinking-2507-stf-w-reasoning-trace-modal.ipynb --
# update DATASET_VARIANT-suffix to match whichever run you're evaluating (e.g.
# "...-adapter-pa", "...-adapter-v6_en").
EVAL_TAG = "pa"
ADAPTER_DIR = f"/mnt/qwen3-4b-thinking-2507-sft/qwen3-4b-thinking-2507-vinumqa-sft-adapter-{EVAL_TAG}"

if not os.path.isdir(ADAPTER_DIR):
    # Fallback: search the Volume (and /root, in case it was copied there) at any
    # nesting depth, in case the path above doesn't match this session's layout.
    _candidates = (
        glob.glob(f"/mnt/qwen3-4b-thinking-2507-sft/**/qwen3-4b-thinking-2507-vinumqa-sft-adapter-{EVAL_TAG}", recursive=True)
        or glob.glob(f"/mnt/qwen3-4b-thinking-2507-sft/**/*adapter*{EVAL_TAG}*", recursive=True)
        or glob.glob(f"/root/**/*adapter*{EVAL_TAG}*", recursive=True)
    )
    if _candidates:
        ADAPTER_DIR = _candidates[0]

assert os.path.isdir(ADAPTER_DIR), (
    f"{ADAPTER_DIR} not found -- check the Volume is attached (sidebar), the training run actually "
    f"reached the Save cell before it stopped, and EVAL_TAG matches the DATASET_VARIANT it trained on, "
    f"or set ADAPTER_DIR manually to the folder you can see in the Volume's file browser."
)
print("Found adapter:", ADAPTER_DIR, "->", sorted(os.listdir(ADAPTER_DIR))[:6])

# from_pretrained on an adapter directory pulls the base model automatically and
# applies the LoRA weights on top.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = ADAPTER_DIR,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)
print("Adapter loaded.")

### Data + prompt (identical to the training notebook)

In [ ]:
import pandas as pd
import glob, os
from pathlib import Path
from tabulate import tabulate

# test.json uploaded manually via the Modal Server Web UI file browser to /root/,
# same pattern as every other Modal notebook in this repo -- adjust _CANDIDATES if
# you put it somewhere else.
_CANDIDATES = [
    Path("/root"),
    Path("/mnt/qwen3-4b-thinking-2507-sft"),
    Path("datasets/ViNumQA"),  # local repo path, if running outside Modal
]
TEST_DIR = next((p for p in _CANDIDATES if (p / "test.json").exists()), None)
if TEST_DIR is None:
    _hits = glob.glob("/root/**/test.json", recursive=True)
    if _hits:
        TEST_DIR = Path(_hits[0]).parent
assert TEST_DIR is not None, "test.json not found -- upload it to /root/, or update _CANDIDATES above."
TEST_JSON_PATH = TEST_DIR / "test.json"

print("Using test.json:", TEST_JSON_PATH)

test_df = pd.read_json(TEST_JSON_PATH)
print(f"test={len(test_df)}")

In [ ]:
def formatting_pre_text(sample):
    return "\n".join(sample["pre_text"])

def formatting_table(sample):
    return tabulate(sample["table"][1:], headers=sample["table"][0], tablefmt="github")

def formatting_post_text(sample):
    return "\n".join(sample["post_text"])

def processing_input_question(sample):
    return sample["qa"]["question"]

def processing_reasoning_trace(sample):
    # None for samples without a verified trace -- kept as None (not "")
    # so build_conversation can tell the two cases apart.
    return sample["qa"].get("reasoning_trace")

def processing_program_content(sample):
    return sample["qa"]["program"]

def processing_answer_content(sample):
    return sample["qa"]["exe_ans"]

def process_split(df):
    df = df.copy()
    df["pre_text_processed"] = df.apply(formatting_pre_text, axis=1)
    df["post_text_processed"] = df.apply(formatting_post_text, axis=1)
    df["table_processed"] = df.apply(formatting_table, axis=1)
    # Keep the raw list-of-rows alongside the markdown rendering: the
    # scorer needs it to resolve programs that name a table row.
    df["table_raw"] = df["table"]
    df["input_question"] = df.apply(processing_input_question, axis=1)
    df["program_processed"] = df.apply(processing_program_content, axis=1)
    df["answer_processed"] = df.apply(processing_answer_content, axis=1)
    df = df[["pre_text_processed", "table_processed", "table_raw", "post_text_processed",
             "input_question", "program_processed", "answer_processed"]]
    df.columns = ["pre_text", "table", "table_raw", "post_text", "question",
                  "program", "answer"]
    return df

test_df = process_split(test_df)
test_df["generated_program"] = ""
test_df.head(3)

In [ ]:
SYSTEM_MESSAGE = """You are a financial analysis AI. Your task is to generate a sequential computation program to answer the question, based on the provided context.

### LIST OF 10 VALID OPERATORS:

1. add(a, b) -> a + b
2. subtract(a, b) -> a - b
3. multiply(a, b) -> a * b
4. divide(a, b) -> a / b
5. exp(a, b) -> a^b
6. greater(a, b) -> 1.0 if a > b, else 0.0
7. table_sum(row_name, none) -> sum of the numeric values in the table row named `row_name`
8. table_average(row_name, none) -> arithmetic mean of the numeric values in the table row named `row_name`
9. table_max(row_name, none) -> maximum of the numeric values in the table row named `row_name`
10. table_min(row_name, none) -> minimum of the numeric values in the table row named `row_name`

### RULES:
- Do not use free-form mathematical symbols ("+", "-", "*", "/") outside of parentheses. Every calculation must use one of the 10 operators above.
- table_* operators take exactly two arguments: the row name (copied exactly as it appears as the first cell of the target row) and the literal `none` (e.g. table_max(Lãi ròng, none)), never a list of numeric values.
- Do not perform mental calculations or provide explanations. The output must contain only the program string.
- Reference the result of a previous step using #0 (step 1), #1 (step 2), etc. Steps are separated by commas.
- Preserve the original number format from the context. If a value is missing, use 'none'."""

USER_MESSAGE_FRAME = """### CONTEXT:
[TEXT BEFORE TABLE]
{pre_text}

[TABLE]
{table}

[TEXT AFTER TABLE]
{post_text}

### QUESTION:
{question}

### PROGRAM:"""

# Same prompt format as the 0-shot/1-shot/sft notebooks, so results are
# comparable across all experiments (only the training regime differs).

### PA / EA

In [ ]:
# ViNumQA scorer -- kept in sync with notebooks/vinumqa/scorer.py, inlined here
# because a Kaggle notebook cannot import from the repository.
#
"""ViNumQA scorer: the FinQA evaluation protocol, adapted to this dataset.

The shared-task paper states that "the official evaluation protocol proposed by
Chen et al. (2021) is adopted", so the semantics here follow `evaluate/evaluate.py`
(FinQA's own script) rather than being reinvented:

  * Program Accuracy is *symbolic* equivalence, via sympy, between the gold and
    predicted expressions -- not a string or structural match. A prediction may
    reorder or restructure the arithmetic, but it may only use literals that
    appear in the gold program, so it cannot invent constants such as the `100`
    of a percentage rescaling.
  * Execution Accuracy compares the executed result to `exe_ans` exactly, after
    rounding to 5 decimals. No tolerance.
  * `greater` yields the strings "yes"/"no", matching how the dataset stores
    those answers.
  * Every step takes exactly two arguments. Verified against the data: all 663
    steps across the gold programs are binary, and `table_*` always takes a row
    label plus `none` (454 occurrences) rather than a list of values (2).
  * FinQA's `const_` tokens are still understood.

Five corrections are applied, each because the unmodified script cannot
reproduce ViNumQA's own gold, not because the protocol was thought wrong:

1. Tokenisation of bracketed row labels. `program_tokenization` splits on every
   bracket, so `table_min(ROE (%), none)` shatters into six tokens and fails the
   four-tokens-per-step structure check. 35 of the 497 test programs name a row
   whose label contains brackets -- `ROE (%)`, `EPS (VND)`, `P/E (x)` -- and all
   35 were unscoreable. Tokenisation is now bracket-depth aware.

2. Accounting negatives. Tables write negative amounts as `(3344)`. The original
   `process_row` takes the text before the first bracket, leaving an empty
   string, so the cell fails to parse. The dataset's own `exe_ans` was computed
   with those values -- e.g. `table_min(LN hoạt động (tỷ đồng), none)` expects
   -3344 from a row holding `(3344)`. The `-1046 ( 1046 )` form the original
   handled correctly is unchanged.

3. Unparseable cells no longer void the whole row. Measured over the 393 gold
   `table_*(<row>, none)` programs in train, skipping such cells reproduces
   `exe_ans` for 386 against 381 when the row is voided, so skipping is what the
   dataset was built with.

4. `exe_ans` is stored as a string here ("31.0") where FinQA stores a number, so
   the comparison `exe_res == gold_res` was never true. It is coerced, leaving
   the "yes"/"no" answers alone.

5. The `assert exe_res == gold_res` inside the program-accuracy branch is
   dropped. It is a debug check, and a single rounding disagreement aborts the
   whole evaluation.

`evaluate_result_official` runs the unmodified protocol for comparison, so the
cost of each correction can be seen rather than assumed.
"""

import re
from typing import List, Optional, Sequence, Tuple, Union

from sympy import simplify

ALL_OPS = ["add", "subtract", "multiply", "divide", "exp", "greater",
           "table_max", "table_min", "table_sum", "table_average"]

_PAREN_NEG_RE = re.compile(r"^\(\s*([\d.,]+)\s*\)$")
_NAME_RE = re.compile(r"\s*([a-zA-Z_]+)\(")


# ------------------------------------------------------------------ numbers --
def str_to_num(text: str) -> Union[float, str]:
    """FinQA's literal parser, unchanged: returns "n/a" rather than raising."""
    text = str(text).replace(",", "")
    try:
        return float(text)
    except ValueError:
        if "%" in text:
            try:
                return float(text.replace("%", "")) / 100.0
            except ValueError:
                return "n/a"
        if text.endswith(("x", "X")):
            # Multiples are written "14.3x" in these tables. Unlike "%", the
            # suffix carries no scaling -- the gold answer for such a row is the
            # plain multiple.
            try:
                return float(text[:-1])
            except ValueError:
                pass
        if "const" in text:
            text = text.replace("const_", "")
            if text == "m1":
                text = "-1"
            try:
                return float(text)
            except ValueError:
                return "n/a"
        return "n/a"


def _cell_to_num(raw: str) -> Union[float, str]:
    """Parse one table cell.

    Adds the `(3344)` form to what the original handled; `$ -1046 ( 1046 )`
    still resolves through the original's "text before the first bracket" rule.
    """
    text = str(raw).replace("$", "").strip()
    m = _PAREN_NEG_RE.match(text)
    if m:
        value = str_to_num(m.group(1))
        return -value if value != "n/a" else "n/a"
    return str_to_num(text.split("(")[0].strip())


_MISSING_CELL_MARKERS = {"", "-", "–", "—", "na", "n/a", "nan", "none"}


def process_row(row_in: Sequence[str]):
    """Numeric values of a table row, or "n/a" if the row cannot be reduced.

    A cell that merely marks a missing period ("-", "NA", an em dash) is
    skipped: rows in this dataset routinely lack a year or two, and voiding the
    whole row over one gap loses reductions the gold answers depend on. A cell
    with real but unreadable content still voids the row, so genuine parse
    failures are not silently averaged away.
    """
    row_out = []
    for cell in row_in:
        text = str(cell).replace("$", "").strip()
        if text.lower() in _MISSING_CELL_MARKERS:
            continue
        num = _cell_to_num(text)
        if num == "n/a":
            return "n/a"
        row_out.append(num)
    return row_out or "n/a"


# -------------------------------------------------------------- tokenisation --
def program_tokenization(original_program: str) -> List[str]:
    """Tokenise into ['op(', arg1, arg2, ')', ..., 'EOF'].

    Bracket-depth aware, so a row label like `ROE (%)` stays one token. The
    original split on every bracket, which shattered such labels and broke the
    four-tokens-per-step structure the rest of the protocol relies on.

    Raises ValueError if trailing, non-whitespace text remains once no further
    step can be parsed (e.g. a step missing its closing paren, which happens
    both in a handful of gold programs and -- more importantly -- in model
    generations cut off by a max_new_tokens limit). An earlier version of this
    tokenizer silently stopped and returned only the steps parsed so far,
    which let a truncated program like "subtract(100, 50), divide(#0, 5"
    (missing text and closing paren) score as a valid, complete one-step
    program instead of being rejected -- a false positive for exactly the kind
    of generation failure this evaluator needs to catch.
    """
    text = str(original_program).strip()
    program: List[str] = []
    pos = 0

    while pos < len(text):
        m = _NAME_RE.match(text, pos)
        if not m:
            break
        open_idx = m.end() - 1

        depth, close = 0, -1
        for i in range(open_idx, len(text)):
            if text[i] == "(":
                depth += 1
            elif text[i] == ")":
                depth -= 1
                if depth == 0:
                    close = i
                    break
        if close == -1:
            raise ValueError(
                f"Unbalanced parentheses (no matching ')' found) in program: '{original_program}'"
            )

        program.append(m.group(1) + "(")
        # Split arguments on depth-0 commas so brackets inside a label survive.
        args, arg_depth, current = [], 0, []
        for ch in text[m.end():close]:
            if ch == "(":
                arg_depth += 1
                current.append(ch)
            elif ch == ")":
                arg_depth -= 1
                current.append(ch)
            elif ch == "," and arg_depth == 0:
                args.append("".join(current).strip())
                current = []
            else:
                current.append(ch)
        if current:
            args.append("".join(current).strip())

        program.extend(args)
        program.append(")")
        pos = close + 1
        while pos < len(text) and text[pos] in ", ":
            pos += 1

    if pos < len(text) and text[pos:].strip():
        raise ValueError(
            f"Trailing unparsed content in program: '{text[pos:]}' (from: '{original_program}')"
        )

    program.append("EOF")
    return program


def extract_program(raw_text: str) -> str:
    """Recover a program string from raw model output.

    Bracket matched, so an outer call is never silently discarded: the earlier
    regex could only match a bracket-free call, so `multiply(divide(a, b), 100)`
    was reduced to its inner `divide(a, b)` and a percentage rescaling scored as
    if it were the gold answer.
    """
    text = re.sub(r"```[a-zA-Z]*", "", str(raw_text)).replace("```", "").strip()

    calls, pos = [], 0
    while pos < len(text):
        m = _NAME_RE.search(text, pos)
        if not m:
            break
        if m.group(1) not in ALL_OPS:
            pos = m.end()
            continue
        depth, close = 0, -1
        for i in range(m.end() - 1, len(text)):
            if text[i] == "(":
                depth += 1
            elif text[i] == ")":
                depth -= 1
                if depth == 0:
                    close = i
                    break
        if close == -1:
            break
        calls.append(text[m.start(1):close + 1].strip())
        pos = close + 1

    return ", ".join(calls) if calls else text



def _steps_from_tokens(program: List[str]) -> List[Tuple[str, str, str]]:
    """Group a tokenised program into (op, arg1, arg2) triples.

    The original walked the token list by joining it and splitting on ")", which
    silently mis-splits any argument containing a bracket -- exactly the row
    labels this dataset uses, e.g. `EPS (VND)`. Grouping the tokens directly is
    equivalent for well-formed programs and correct for those.
    """
    body = program[:-1] if program and program[-1] == "EOF" else list(program)
    if len(body) % 4 != 0:
        raise ValueError("token count is not a multiple of four")
    steps = []
    for i in range(0, len(body), 4):
        op_token, arg1, arg2, close = body[i:i + 4]
        if not op_token.endswith("(") or close != ")":
            raise ValueError("malformed step")
        op = op_token[:-1].strip()
        if op not in ALL_OPS:
            raise ValueError(f"unknown operator {op!r}")
        steps.append((op, arg1.strip(), arg2.strip()))
    return steps

# ---------------------------------------------------------------- execution --
def eval_program(program: List[str], table: Optional[Sequence[Sequence[str]]]):
    """Execute a tokenised program. Returns (invalid_flag, result)."""
    this_res: Union[float, str] = "n/a"

    try:
        steps = _steps_from_tokens(program)
        res_dict = {}

        for ind, (op, arg1, arg2) in enumerate(steps):
            if op in ("add", "subtract", "multiply", "divide", "exp", "greater"):
                if "#" in arg1:
                    arg1 = res_dict[int(arg1.replace("#", ""))]
                else:
                    arg1 = str_to_num(arg1)
                    if arg1 == "n/a":
                        return 1, "n/a"
                if "#" in arg2:
                    arg2 = res_dict[int(arg2.replace("#", ""))]
                else:
                    arg2 = str_to_num(arg2)
                    if arg2 == "n/a":
                        return 1, "n/a"

                if op == "add":
                    this_res = arg1 + arg2
                elif op == "subtract":
                    this_res = arg1 - arg2
                elif op == "multiply":
                    this_res = arg1 * arg2
                elif op == "divide":
                    this_res = arg1 / arg2
                elif op == "exp":
                    this_res = arg1 ** arg2
                else:
                    this_res = "yes" if arg1 > arg2 else "no"

            else:  # table_*
                table_dict = {row[0]: row[1:] for row in (table or [])}
                if "#" in arg1:
                    num_row = [res_dict[int(arg1.replace("#", ""))]]
                else:
                    if arg1 not in table_dict:
                        return 1, "n/a"
                    num_row = process_row(table_dict[arg1])
                if num_row == "n/a":
                    return 1, "n/a"

                if op == "table_max":
                    this_res = max(num_row)
                elif op == "table_min":
                    this_res = min(num_row)
                elif op == "table_sum":
                    this_res = sum(num_row)
                else:
                    this_res = sum(num_row) / len(num_row)

            res_dict[ind] = this_res

        if this_res not in ("yes", "no", "n/a"):
            this_res = round(this_res, 5)
    except Exception:
        return 1, "n/a"

    return 0, this_res


# ------------------------------------------------------------------ program --
def equal_program(program1: List[str], program2: List[str]) -> bool:
    """Symbolic equivalence of gold (program1) and prediction (program2).

    Same protocol as the official implementation -- literals become symbols,
    table steps become opaque variables, and the two expressions are compared
    after `simplify`, so a differently-arranged but algebraically identical
    program still counts. A prediction may only use symbols that appear in gold,
    which is what stops it from introducing a constant of its own (the `100` of
    a percentage rescaling, say). Only the step-splitting differs: it groups
    tokens rather than splitting a joined string on ")".
    """
    try:
        steps1 = _steps_from_tokens(program1)
    except Exception:
        return False

    sym_map, sym_ind = {}, 0
    for op, arg1, arg2 in steps1:
        if "table" in op:
            key = (op, arg1, arg2)
            if key not in sym_map:
                sym_map[key] = "a" + str(sym_ind)
                sym_ind += 1
        else:
            for arg in (arg1, arg2):
                if "#" not in arg and arg not in sym_map:
                    sym_map[arg] = "a" + str(sym_ind)
                    sym_ind += 1

    try:
        steps2 = _steps_from_tokens(program2)
    except Exception:
        return False

    for ind, (op, arg1, arg2) in enumerate(steps2):
        if "table" in op:
            if (op, arg1, arg2) not in sym_map:
                return False
        else:
            for arg in (arg1, arg2):
                if "#" not in arg:
                    if arg not in sym_map:
                        return False
                elif int(arg.strip("#")) >= ind:
                    return False

    def symbol_recur(ind, steps):
        op, arg1, arg2 = steps[ind]
        if "table" in op:
            return sym_map[(op, arg1, arg2)]
        parts = []
        for arg in (arg1, arg2):
            if "#" in arg:
                parts.append(symbol_recur(int(arg.replace("#", "")), steps))
            else:
                parts.append(sym_map[arg])
        sign = {"add": "+", "subtract": "-", "multiply": "*",
                "divide": "/", "exp": "**", "greater": ">"}[op]
        return f"( {parts[0]} {sign} {parts[1]} )"

    try:
        sym1 = simplify(symbol_recur(len(steps1) - 1, steps1), evaluate=False)
        sym2 = simplify(symbol_recur(len(steps2) - 1, steps2), evaluate=False)
    except Exception:
        return False

    return sym1 == sym2


# ------------------------------------------------------------------ metrics --
def _coerce_answer(value):
    """ViNumQA stores exe_ans as a string; "yes"/"no" stay as they are."""
    try:
        return float(value)
    except (TypeError, ValueError):
        return value


def score_one(generated_program: str, gold_program: str, gold_answer,
              table: Optional[Sequence[Sequence[str]]] = None,
              extract_first: bool = True) -> Tuple[float, float]:
    """(program_accuracy, execution_accuracy) for a single item.

    A generated_program that fails to tokenize (e.g. cut off mid-generation,
    missing a closing paren) scores (0.0, 0.0) rather than raising -- this is
    expected input from a real model, not a bug to surface as an exception.
    gold_program is assumed well-formed and is not caught the same way, so a
    malformed *gold* label still raises loudly instead of silently scoring 0.
    """
    generated = extract_program(generated_program) if extract_first else generated_program
    gold_tok = program_tokenization(gold_program)
    gold_res = _coerce_answer(gold_answer)

    try:
        pred_tok = program_tokenization(generated)
    except ValueError:
        return 0.0, 0.0

    invalid, exe_res = eval_program(pred_tok, table)
    ea = 1.0 if invalid == 0 and exe_res == gold_res else 0.0

    try:
        pa = 1.0 if equal_program(gold_tok, pred_tok) else 0.0
    except Exception:
        pa = 0.0

    return pa, ea


def evaluate_dataframe(df, generated_col: str = "generated_program",
                       gold_program_col: str = "program",
                       gold_answer_col: str = "answer",
                       table_col: str = "table_raw",
                       extract_first: bool = True):
    """Score a DataFrame, returning (df + per-row scores, summary).

    `table_col` must hold the raw table (list of rows); without it, programs
    naming a table row cannot execute and score 0 on EA.
    """
    df = df.copy()
    pa_scores, ea_scores = [], []

    for _, row in df.iterrows():
        table = row[table_col] if table_col in df.columns else None
        pa, ea = score_one(row[generated_col], row[gold_program_col],
                           row[gold_answer_col], table, extract_first)
        pa_scores.append(pa)
        ea_scores.append(ea)

    df["pa_score"] = pa_scores
    df["ea_score"] = ea_scores
    return df, {
        "program_accuracy": sum(pa_scores) / len(pa_scores) if pa_scores else 0.0,
        "execution_accuracy": sum(ea_scores) / len(ea_scores) if ea_scores else 0.0,
    }

In [ ]:
import gc
import glob, os
from pathlib import Path
from tqdm import tqdm

QWEN3_THINK_END_TOKEN_ID = 151668  # "</think>"

# One prompt at a time, deliberately. Batching several prompts together requires
# left padding, and Unsloth's fast inference path derives token positions from
# the cache length rather than from the attention mask -- so padded rows decode
# at the wrong positions. Measured on the wo-reasoning model, that cost 0.6419
# -> 0.6338 PA. Extra votes come from num_return_sequences instead: those share
# a single prompt, so there is nothing to pad, and memory stays bounded by
# N_VOTES rather than by N_VOTES x batch.
#
# N_VOTES = 1 is the like-for-like number. Raising it multiplies the runtime,
# and this model emits a full reasoning trace per sample (~24s each on a T4,
# so ~3.3h for one pass over the 497 test questions) -- k=5 would not fit a
# session. Run k=1 first; the checkpoint below lets a later pass resume.
N_VOTES = 1
# 2048, matching the run this adapter came from. Too low a cap is not a mild
# loss: if generation is cut off before the model closes </think>, strip_think
# finds no closing tag and hands the whole reasoning trace to the parser as if
# it were the program, so a sample that would have been right scores 0 on both
# metrics. Generation stops at EOS anyway, so the higher cap costs nothing on
# samples that finish early.
EVAL_MAX_NEW_TOKENS = 2048
TEMPERATURE = 0.6   # only used when N_VOTES > 1; Qwen3 thinking defaults
TOP_P = 0.95

CKPT_PATH = Path("/mnt/qwen3-4b-thinking-2507-sft") / f"eval_partial_k{N_VOTES}_{EVAL_TAG}.csv"  # Volume, tagged with EVAL_TAG so a restart can resume without colliding across adapters

if CKPT_PATH.exists():
    _done = pd.read_csv(CKPT_PATH, index_col=0).fillna("")
    test_df.loc[_done.index, "generated_program"] = _done["generated_program"].values
    print(f"Resumed {(test_df['generated_program'] != '').sum()} / {len(test_df)} from Volume checkpoint.")
else:
    print("No checkpoint found on the Volume -- starting fresh.")


def build_prompt(row):
    user_msg = USER_MESSAGE_FRAME.format(
        pre_text=row["pre_text"], table=row["table"],
        post_text=row["post_text"], question=row["question"],
    )
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": user_msg},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )


def strip_think(output_ids):
    """Keep only what follows the final </think>; the model always emits one."""
    ids = list(output_ids)
    try:
        cut = len(ids) - ids[::-1].index(QWEN3_THINK_END_TOKEN_ID)
    except ValueError:
        cut = 0
    return tokenizer.decode(ids[cut:], skip_special_tokens=True).strip()


def vote(candidates):
    """Most common program among candidates, keyed by normalised form so that
    cosmetically different but structurally identical programs share a vote."""
    cleaned = [extract_program(c) for c in candidates if c and c.strip()]
    if not cleaned:
        return ""
    keyed = {}
    for c in cleaned:
        try:
            key = str(program_tokenization(c))
        except Exception:
            key = c.strip()
        keyed.setdefault(key, []).append(c)
    best = max(keyed, key=lambda k: (len(keyed[k]), -cleaned.index(keyed[k][0])))
    return keyed[best][0]


n_unclosed = 0
todo = [i for i in test_df.index if not str(test_df.at[i, "generated_program"]).strip()]

for n, df_index in enumerate(tqdm(todo, desc=f"Generating (k={N_VOTES})")):
    enc = tokenizer([build_prompt(test_df.loc[df_index])], return_tensors="pt").to(model.device)

    gen_kwargs = dict(max_new_tokens=EVAL_MAX_NEW_TOKENS)
    if N_VOTES > 1:
        gen_kwargs.update(do_sample=True, temperature=TEMPERATURE, top_p=TOP_P,
                          num_return_sequences=N_VOTES)

    try:
        with torch.no_grad():
            out = model.generate(**enc, **gen_kwargs)
        prompt_len = enc["input_ids"].shape[1]
        gen_only = [r[prompt_len:].tolist() for r in out]
        # A generation that never closed </think> was almost certainly cut off
        # by the token cap; count them so a too-low cap is visible rather than
        # silently scoring zeros.
        n_unclosed += sum(1 for g in gen_only if QWEN3_THINK_END_TOKEN_ID not in g)
        cands = [strip_think(g) for g in gen_only]
        test_df.at[df_index, "generated_program"] = vote(cands) if N_VOTES > 1 else cands[0]
        del enc, out
    except torch.cuda.OutOfMemoryError:
        print(f"  OOM on index {df_index}; leaving it blank (scored 0).")
        test_df.at[df_index, "generated_program"] = ""

    if n % 25 == 0:
        test_df[["generated_program"]].to_csv(CKPT_PATH)
        gc.collect()
        torch.cuda.empty_cache()

test_df[["generated_program"]].to_csv(CKPT_PATH)
print(f"Done. {(test_df['generated_program'] != '').sum()} / {len(test_df)} generated "
      f"(N_VOTES={N_VOTES}, {len(todo)} newly generated this run).")
print(f"Generations that never closed </think>: {n_unclosed} "
      f"-- these were cut off by EVAL_MAX_NEW_TOKENS and score 0; "
      f"raise the cap if this is not near zero.")

In [ ]:
df_scored, summary = evaluate_dataframe(test_df)
print(summary)

# The earlier figures came from a scorer that credited some wrong answers and
# rejected some right ones, so they are shown for continuity rather than as a
# like-for-like comparison. What changed:
#   * an outer call is no longer discarded, so `multiply(divide(a, b), 100)` is
#     no longer scored as if it were `divide(a, b)`;
#   * Execution Accuracy compares exactly after rounding to 5 decimals, as the
#     FinQA protocol does, instead of accepting a 1e-3 relative error;
#   * programs naming a table row now execute, and row labels containing
#     brackets -- `ROE (%)`, `EPS (VND)` -- parse instead of scoring 0.
# Feeding the gold programs back through this scorer returns 1.0/1.0 on both
# test and valid, so nothing correct is being turned away.
PREVIOUS = {"program_accuracy": 0.6599, "execution_accuracy": 0.6680}

print()
print(f"{'scorer':<34}{'PA':>10}{'EA':>10}")
print("-" * 54)
print(f"{'previous (repo scorer)':<34}{PREVIOUS['program_accuracy']:>10.4f}"
      f"{PREVIOUS['execution_accuracy']:>10.4f}")
print(f"{'corrected (FinQA protocol)':<34}{summary['program_accuracy']:>10.4f}"
      f"{summary['execution_accuracy']:>10.4f}")